# Aula 11 - Notebook: Modelagem Topológica de Tubulações como Dígrafos Ponderados

Neste notebook implementamos a classe base `GrafoTubulacao` para representar a malha hidráulica do complexo químico de fertilizantes como um Grafo Dirigido e Ponderado $G=(V, E, W)$.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

from typing import List, Dict, Tuple, Any

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float, 
                           tag_valvula: str, diametro_pol: float = 4.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        
        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Válvula ISA": tag_valvula,
            "Diâmetro (pol)": diametro_pol
        })

    def obter_graus(self) -> List[Dict[str, Any]]:
        graus = []
        for i, v in enumerate(self.vertices):
            deg_out = sum(self.adj_binaria[i])
            deg_in = sum(self.adj_binaria[r][i] for r in range(self.n))
            graus.append({"Equipamento": v, "Grau Entrada (deg-)": deg_in, "Grau Saída (deg+)": deg_out})
        return graus

nos_processo = ["TK-301_NH3", "TK-302_H3PO4", "MAN-101", "P-101", "P-102", "R-101", "TK-303_Pulmao", "GRAN-201"]
rede = GrafoTubulacao(nos_processo)

rede.adicionar_tubulacao("TK-301_NH3", "MAN-101", 15.0, "XV-301", 3.0)
rede.adicionar_tubulacao("TK-302_H3PO4", "MAN-101", 12.0, "XV-302", 4.0)
rede.adicionar_tubulacao("MAN-101", "P-101", 8.0, "XV-101A", 4.0)
rede.adicionar_tubulacao("MAN-101", "P-102", 10.0, "XV-101B", 4.0)
rede.adicionar_tubulacao("P-101", "R-101", 25.0, "XV-102A", 4.0)
rede.adicionar_tubulacao("P-102", "R-101", 22.0, "XV-102B", 4.0)
rede.adicionar_tubulacao("R-101", "GRAN-201", 30.0, "XV-201", 6.0)
rede.adicionar_tubulacao("R-101", "TK-303_Pulmao", 18.0, "XV-202", 6.0)
rede.adicionar_tubulacao("TK-303_Pulmao", "GRAN-201", 20.0, "XV-203", 6.0)

print("Tabela de Dutos de Processo:")
print(formatar_tabela(rede.arestas_detalhes))
print("\n--- Graus Topológicos ---")
print(formatar_tabela(rede.obter_graus()))
print("\n--- Matriz de Adjacência Ponderada (Metros) ---")
print(formatar_matriz(rede.adj_pesos, rede.vertices, rede.vertices))


Tabela de Dutos de Processo:
Origem        | Destino       | Comprimento (m) | Válvula ISA | Diâmetro (pol)
--------------+---------------+-----------------+-------------+---------------
TK-301_NH3    | MAN-101       | 15.0            | XV-301      | 3.0           
TK-302_H3PO4  | MAN-101       | 12.0            | XV-302      | 4.0           
MAN-101       | P-101         | 8.0             | XV-101A     | 4.0           
MAN-101       | P-102         | 10.0            | XV-101B     | 4.0           
P-101         | R-101         | 25.0            | XV-102A     | 4.0           
P-102         | R-101         | 22.0            | XV-102B     | 4.0           
R-101         | GRAN-201      | 30.0            | XV-201      | 6.0           
R-101         | TK-303_Pulmao | 18.0            | XV-202      | 6.0           
TK-303_Pulmao | GRAN-201      | 20.0            | XV-203      | 6.0           

--- Graus Topológicos ---
Equipamento   | Grau Entrada (deg-) | Grau Saída (deg+)
--------------+----